# nmtc-application-builder — Quickstart

**Week 1 of 4: Foundation & Pipeline Intelligence**

This notebook demonstrates the complete Week 1 workflow:
1. Load a sample CDE profile
2. Load a sample pipeline (20 projects)
3. Run `app.analyze()`
4. Print the rich summary
5. Inspect the readiness score

All calls work offline — external APIs fall back to sample data automatically.

In [1]:
import sys
sys.path.insert(0, '..')  # add repo root to path

from nmtcapp.core.application import Application
from nmtcapp.core.cde import CDEProfile
from nmtcapp.core.pipeline import Pipeline

## Step 1 — Load a Sample CDE Profile

In [2]:
cde = CDEProfile.sample()

print(f"CDE:               {cde.name}")
print(f"CDE ID:            {cde.cde_id}")
print(f"Certified:         {cde.certification_date}")
print(f"Target Markets:    {', '.join(cde.target_markets)}")
print(f"Prior Awards:      {len(cde.prior_awards)} rounds, ${cde.total_prior_allocation():,.0f} total")
print(f"\nMission: {cde.mission[:120]}...")

CDE:               Heartland Impact CDE, LLC
CDE ID:            CDE-2018-0117
Certified:         2018-06-20
Target Markets:    Illinois, Ohio, Michigan, Indiana, Tennessee, Georgia, Louisiana, Missouri
Prior Awards:      3 rounds, $155,000,000 total

Mission: Deploy New Markets Tax Credit capital into deep-distress communities across the Midwest and South, with a focus on healt...


## Step 2 — Load a Sample Pipeline (20 projects)

In [3]:
pipeline = Pipeline.sample(n=20)

print(f"Pipeline: {len(pipeline)} projects")
print(f"Total QEI: ${sum(p.qei_request for p in pipeline):,.0f}")
print()

df = pipeline.to_dataframe()
display(df[['project_name', 'state', 'sector', 'qei_request', 'distress_level', 'expected_jobs_created']].head(10))

Pipeline: 20 projects
Total QEI: $122,500,000



,project_name,state,sector,qei_request,distress_level,expected_jobs_created
0,Southside Community Health Center,IL,healthcare,8500000,deep,52
1,East Houston Charter Academy,TX,education,7000000,deep,38
2,Bronx Food Hub,NY,small_business,5000000,severe,65
3,Watts Manufacturing Center,CA,small_business,4500000,deep,80
4,Cleveland Neighborhood Clinic,OH,healthcare,7500000,deep,42
5,Atlanta Affordable Homes Phase II,GA,affordable_housing,9000000,severe,15
6,Miami Community Resource Center,FL,community_facility,4000000,lic,28
7,North Philadelphia Wellness Hub,PA,healthcare,8000000,deep,55
8,New Orleans Culinary Arts Center,LA,education,6000000,deep,35
9,Memphis Small Business Incubator,TN,small_business,3500000,deep,90


In [4]:
# Pipeline sector and state distribution
print("Sector breakdown:")
print(df['sector'].value_counts().to_string())
print()
print("State breakdown:")
print(df['state'].value_counts().to_string())

Sector breakdown:
sector
healthcare            5
small_business        4
education             3
affordable_housing    3
community_facility    2
mixed_use             2
clean_energy          1

State breakdown:
state
IL    1
TX    1
MD    1
KS    1
MS    1
IN    1
MI    1
AZ    1
NC    1
MO    1
WI    1
TN    1
LA    1
PA    1
FL    1
GA    1
OH    1
CA    1
NY    1
NM    1


## Step 3 — Create Application and Run analyze()

In [5]:
app = Application(
    cde=cde,
    requested_allocation=65_000_000,
    application_round="CY2025",
)
app.add_pipeline(pipeline)

print("Running analysis...")
analysis = app.analyze()
print("Done.")

Running analysis...
Done.


## Step 4 — Print the Rich Summary

In [6]:
analysis.summary()


  NMTC APPLICATION ANALYSIS
  CDE:   Heartland Impact CDE, LLC
  Round: CY2025  |  Requested: $65,000,000
  Analyzed: 2026-05-08T23:11:00
PIPELINE ANALYSIS SUMMARY
  Projects:        20
  Total QEI:       $   122,500,000
  Total Cost:      $   170,600,000
  Eligible:        100%

── Distress Concentration ─────────────────────────────
  Deep/Severe:     87%  (✓ target)
  LIC:             13%
  Non-LIC:         0%
  Native Area:     10%
  Historical Rank: top_quartile

── Geographic Diversity ────────────────────────────────
  States:          20
  MSAs:            20
  Urban/Rural:     86% / 14%
  HHI:             549  (diverse)

── Sector Mix ──────────────────────────────────────────
  Sectors:         7
  Dominant:        healthcare
  High Priority:   65%
  Diversity Score: 91.8/100

── Impact Projections ──────────────────────────────────
  Jobs Created:    864
  Jobs Retained:   298
  Units Built:     234
  Sq Ft:           198,500
  Jobs/$MM QEI:    7.0
  Benchmark:       averag

## Step 5 — Inspect the Readiness Score

In [7]:
score = analysis.readiness_score
print(score.summary())

  APPLICATION READINESS SCORE: 86.6/100  [A]
  [█████████████████████████░░░░░] 86.6%

Component Scores:
  Eligibility Quality            100.0/100
  Distress Concentration         100.0/100
  Geographic Diversity           100.0/100
  Impact Metrics                  38.2/100
  Validation Pass Rate           100.0/100
  Completeness                    80.0/100

Top Strengths:
  + High pipeline eligibility rate (≥80% score)
  + Strong deep/severe distress concentration
  + Good geographic diversity across multiple states

Areas for Improvement:
  - Jobs/impact per million QEI below CDFI Fund average

Recommendations:
  → Add operating business projects (manufacturing, healthcare) to improve jobs-per-million-QEI metric above CDFI Fund average of 12


In [8]:
# Component scores as a DataFrame
import pandas as pd

components_df = pd.DataFrame.from_dict(
    score.component_scores, orient='index', columns=['Score (0-100)']
)
components_df.index.name = 'Component'
display(components_df.sort_values('Score (0-100)', ascending=False))

,Score (0-100)
Component,
eligibility_quality,100.0
distress_concentration,100.0
geographic_diversity,100.0
validation_pass_rate,100.0
completeness,80.0
impact_metrics,38.2


## Bonus — Distress Concentration Deep Dive

In [9]:
distress = analysis.distress_analysis

print("Distress Concentration Summary")
print("=" * 40)
print(f"  Deep:              {distress['pct_deep']:.1%}")
print(f"  Severe:            {distress['pct_severe']:.1%}")
print(f"  Deep + Severe:     {distress['pct_deep_or_severe']:.1%}")
print(f"  Standard LIC:      {distress['pct_lic']:.1%}")
print(f"  Native Area:       {distress['pct_native_area']:.1%}")
print(f"  High Mig. Rural:   {distress['pct_high_migration_rural']:.1%}")
print()
print(f"  Meets min threshold (50%):    {distress['meets_min_threshold']}")
print(f"  Meets target threshold (75%): {distress['meets_target_threshold']}")
print(f"  vs. Historical Winners:       {distress['vs_historical_winners']}")

Distress Concentration Summary
  Deep:              58.0%
  Severe:            28.7%
  Deep + Severe:     86.7%
  Standard LIC:      13.3%
  Native Area:       10.0%
  High Mig. Rural:   2.6%

  Meets min threshold (50%):    True
  Meets target threshold (75%): True
  vs. Historical Winners:       top_quartile


## Bonus — Loading from CSV

You can also load your pipeline from a CSV file:

In [10]:
# Export the sample pipeline to CSV, then reload it
import tempfile, os

with tempfile.NamedTemporaryFile(mode='w', suffix='.csv', delete=False) as f:
    pipeline.to_dataframe().to_csv(f.name, index=False)
    csv_path = f.name

reloaded = Pipeline.from_csv(csv_path)
print(f"Reloaded {len(reloaded)} projects from CSV")
os.unlink(csv_path)

Reloaded 20 projects from CSV


## What's Next — Week 2

Week 2 will add:
- Word/Excel output — auto-generated narrative sections
- Pro forma tables for each project
- Application section templates (Community Impact, Business Strategy, CDE Capacity)

See the [GitHub repo](https://github.com/Jaypatel1511/nmtc-application-builder) for the full roadmap.